In [0]:
from pyspark.sql.functions import regexp_replace, col, count, when, udf, first
from pyspark.sql.types import DecimalType, StringType

In [0]:
# caminho para o arquivo csv
caminho_csv = "/Workspace/Users/pnf@cesar.school/Grupo7-Setor-de-Seguros/data/raw/seguros_vida_grande.csv"

In [0]:
# lendo o arquivo csv em um dataframe com algumas opções avançadas
df_seguros = spark.read\
    .format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .option("sep", ";")\
    .option("encoding", "ISO-8859-1")\
    .load(caminho_csv)
# visualizando o dataframe
display(df_seguros)

In [0]:
from pyspark.sql.functions import col

def print_mixed_characters():
    colunas = df_seguros.columns
    
    for coluna in colunas:
        if dict(df_seguros.dtypes)[coluna] == 'string':
            print(f"\nMixed character values in column '{coluna}':")
            # Filter and show values that contain both letters AND non-letters/non-spaces
            df_seguros.filter(
                (col(coluna).rlike('[A-Za-z]')) &  # Has at least one letter
                (col(coluna).rlike('[^A-Za-z\\s]')) &  # Has at least one character that's NOT a letter or space
                (col(coluna).isNotNull())
            ).select(coluna).distinct().show(truncate=False)

print_mixed_characters()

In [0]:
from pyspark.sql.functions import col, regexp_replace, trim

# Remove prefixes and trim any resulting extra spaces for both columns
df_seguros = df_seguros.withColumn(
    "nome_contratante",
    trim(
        regexp_replace(
            col("nome_contratante"),
            "^(Dr\\.|Sr\\.|Sra\\.|Srta\\.)\\s+", ""
        )
    )
).withColumn(
    "nome_beneficiario",
    trim(
        regexp_replace(
            col("nome_beneficiario"),
            "^(Dr\\.|Sr\\.|Sra\\.|Srta\\.)\\s+", ""
        )
    )
)

In [0]:
def corrigir_encoding(entrada):
    mapeamento = {"Ç":"ã",                  
                  "Ç½":"â",
                  "Ç":"á",
                  "á¦":"ê",
                  "á¸":"é",
                  "áð":"í",
                  "á":"Í",  
                  "Çð":"í",
                  "Ç§":"ú",
                  "á§":"ú",
                  "áõ":"ç",}

    texto_corrigido = str(entrada)
    for incorreto, correto in mapeamento.items():
        texto_corrigido = texto_corrigido.replace(incorreto, correto)
    
    return texto_corrigido

corrigir_encoding_udf = udf(corrigir_encoding, StringType())

colunas = df_seguros.columns

for coluna in colunas:
    if dict(df_seguros.dtypes)[coluna] == 'string':
        df_seguros = df_seguros.withColumn(coluna, corrigir_encoding_udf(col(coluna)))



In [0]:
display(df_seguros)

In [0]:
colunas_para_converter = [
    "valor_pagamento",
    "valor_premio",
    "Valor Cob 1",
    "Valor Cob 2",
    "Valor Cob 3",
    "capital_segurado"
]

for nome_coluna in colunas_para_converter:
    df_seguros = df_seguros.withColumn(
        nome_coluna,
        regexp_replace(df_seguros[nome_coluna], ",", ".")
        .cast(DecimalType(18, 2))
    )

In [0]:
df_seguros.printSchema()

Tratando dados nulos

In [0]:
contador_nulo = [
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_seguros.columns
]

display(df_seguros.select(contador_nulo))